# Week 12, Day 3: Convexity: when every valley is the bottom

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Test convexity with random chords

Write `chord_violation(f, lo, hi, n=2000)`: draw n random pairs (a, b) in [lo, hi] and random t in [0, 1], and return the largest amount by which f at the interpolated point exceeds the chord, or 0 if it never does. It is a numerical check, not a proof: it can show a function is not convex, never that it is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1951)

def chord_violation(f, lo, hi, n=2000):
    a, b = rng.uniform(lo, hi, n), rng.uniform(lo, hi, n)
    t = rng.uniform(0, 1, n)
    gap = ...                      # f at the interpolated point minus the chord
    return max(0.0, gap.max())

for name, f in [("x^2", lambda x: x ** 2), ("sin", np.sin), ("double well", lambda x: (x ** 2 - 1) ** 2)]:
    print(f"{name:12s} largest violation {chord_violation(f, -2, 2):.4f}")

In [ ]:
import numpy as np
def test_convex():
    "chord_violation finds no violation for x^2, exp and softplus"
    for f in [lambda x: x ** 2, np.exp, lambda x: np.log1p(np.exp(x))]:
        assert chord_violation(f, -3, 3) < 1e-12

def test_nonconvex():
    "it finds violations for sin and the double well"
    assert chord_violation(np.sin, -3, 3) > 0.1
    assert chord_violation(lambda x: (x ** 2 - 1) ** 2, -2, 2) > 0.1

test_convex()
test_nonconvex()
print("All checks passed")

### Exercise: Check the Hessians

For a random design matrix, write the Hessians of the logistic loss and of the Poisson negative log-likelihood (both averaged over T examples). At 200 random weight vectors, record the smallest eigenvalue of each in `min_eig_logistic` and `min_eig_poisson`.

In [ ]:
T = 300
Xd = np.column_stack([rng.normal(size=(T, 2)), np.ones(T)])
def sigmoid(u): return 1 / (1 + np.exp(-u))
def poisson_grad(w):                     # gradient of the mean NLL; counts do not affect the Hessian
    return Xd.T @ (np.exp(Xd @ w) - 1.0) / T

def logistic_hessian(w):
    p = sigmoid(Xd @ w)
    return ...

def poisson_hessian(w):
    lam = np.exp(Xd @ w)
    return ...

min_eig_logistic, min_eig_poisson = [], []
for _ in range(200):
    w = rng.normal(size=3) * 2
    min_eig_logistic.append(np.linalg.eigvalsh(logistic_hessian(w)).min())
    min_eig_poisson.append(np.linalg.eigvalsh(poisson_hessian(w)).min())
print("smallest eigenvalue seen: logistic", f"{min(min_eig_logistic):.2e}", " Poisson", f"{min(min_eig_poisson):.2e}")

In [ ]:
import numpy as np
def test_min_eigs():
    "the smallest Hessian eigenvalue is non-negative at every sampled point, for both losses"
    assert min(min_eig_logistic) > -1e-10 and min(min_eig_poisson) > -1e-10

def test_formula():
    "poisson_hessian matches a numerical Hessian of the Poisson NLL"
    w = np.random.default_rng(9).normal(size=3) * 0.3
    H = np.zeros((3, 3))
    for j in range(3):
        e = np.zeros(3); e[j] = 1e-5
        H[:, j] = (poisson_grad(w + e) - poisson_grad(w - e)) / 2e-5
    assert np.allclose(poisson_hessian(w), H, atol=1e-5)

test_min_eigs()
test_formula()
print("All checks passed")

### Exercise: Two minima and a worse midpoint

Train a two-hidden-unit tanh network to fit y = sin(2x) by gradient descent (code given). Build `w_B` by swapping the two hidden units in the trained weights `w_A` (rows of W1, entries of b1 and w2), and `w_mid`, their average. Compare the three losses.

In [ ]:
x = np.linspace(-2, 2, 100); y = np.sin(2 * x)
def unpack(w): return w[0:2], w[2:4], w[4:6], w[6]        # input weights, biases, output weights, output bias
def net_loss(w):
    W1, b1, w2, b2 = unpack(w)
    h = np.tanh(np.outer(x, W1) + b1)
    return np.mean((h @ w2 + b2 - y) ** 2)
def num_grad(f, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(w + e) - f(w - e)) / (2 * h)
    return g
w_A = np.array([1.5, -0.8, 0.5, 0.3, 1.0, 1.0, 0.0])
for _ in range(3000):
    w_A = w_A - 0.1 * num_grad(net_loss, w_A)

W1, b1, w2, b2 = unpack(w_A)
w_B = ...
w_mid = ...
print(f"loss at w_A {net_loss(w_A):.4f}, at swapped w_B {net_loss(w_B):.4f}, at the midpoint {net_loss(w_mid):.4f}")

In [ ]:
import numpy as np
def test_swap():
    "swapping the hidden units leaves the loss unchanged"
    assert np.isclose(net_loss(w_A), net_loss(w_B))

def test_midpoint():
    "the loss at the midpoint is higher than at both minima, so the loss is not convex"
    assert net_loss(w_mid) > net_loss(w_A) + 0.05

test_swap()
test_midpoint()
print("All checks passed")